# MEMC cvičení 7 – Snižování rozptylu

---

Chyba Monte Carla klesá jako $\sigma/\sqrt N$. Chceme-li desetkrát přesnější výsledek, potřebujeme **stonásobek** vzorků. Existuje druhá cesta: zmenšit **$\sigma$**. Stejnou střední hodnotu totiž lze odhadnout mnoha různými náhodnými veličinami a ty se liší rozptylem.

V sešitu na jednom příkladu, $I = \int_0^1 e^{-x^2}\,dx$ (známe z cvičení 1), porovnáme čtyři triky:

| Trik | Myšlenka |
|------|----------|
| průměr místo trefa/netrefa | použij hodnotu funkce, ne jen informaci „pod / nad" |
| kontrolní proměnná (control variate) | odečti něco, co je podobné a jehož střední hodnotu známe |
| stratifikace | rozlož body rovnoměrněji než čistá náhoda |
| vážení podle důležitosti (importance sampling) | generuj tam, kde na integrandu záleží |

Na konci uvidíte, že u **vzácných jevů** (cvičení 2) je poslední trik rozdíl mezi „odhad nula" a přesným výsledkem.

In [ ]:
import math
import random

import matplotlib.pyplot as plt

random.seed(1)  # opakovatelné výsledky


def estimate(xs):
    '''Průměr a směrodatná chyba z hotového seznamu hodnot.'''
    n = len(xs)
    mean = sum(xs) / n
    var = sum((x - mean) ** 2 for x in xs) / (n - 1)
    return mean, math.sqrt(var / n)


def check(name, mean, se, exact, k=5):
    '''Vypíše odhad a ověří, že leží do k směrodatných chyb od přesné hodnoty.'''
    print(f"{name}: {mean:.6g} +- {se:.3g}  (přesně {exact:.6g})")
    assert abs(mean - exact) <= k * se, name


def Phi(x):
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))


f = lambda x: math.exp(-x * x)
Z = math.sqrt(math.pi) / 2 * math.erf(1)  # přesná hodnota integrálu
print(f"I = {Z:.6f}")

## 1. Co chceme měřit

Odhadovat budeme $N$ nezávislými opakováními, a kvalitu odhadu popisuje **rozptyl jednoho příspěvku** $\sigma^2$: směrodatná chyba je $\sigma/\sqrt N$. Při stejném $N$ je tedy lepší metoda ta s menším $\sigma^2$.

Metody se liší i **cenou** jednoho příspěvku (některé potřebují dvě náhodná čísla, jiné výpočet navíc). Férové srovnání proto sleduje rozptyl při **stejném počtu vyhodnocení funkce**, což budeme dělat.

Rozptyl budeme měřit **experimentem**: každou metodu s $N$ body opakujeme $R$-krát a spočítáme směrodatnou odchylku $R$ odhadů. To je přímo skutečná chyba odhadu (bez teorie, bez předpokladů).

## 2. Průměr místo trefa/netrefa

V cvičení 1 jsme integrál odhadovali dvojím způsobem:

- **trefa/netrefa:** bod $(\gamma_1, \gamma_2)$ v jednotkovém čtverci, $\mathbb 1[\gamma_2 < f(\gamma_1)]$; rozptyl jednoho příspěvku je $I(1-I)$.
- **průměr:** $f(\gamma)$ s $\gamma \sim R(0,1)$; rozptyl jednoho příspěvku je $D f(\gamma) = \int_0^1 f^2 - I^2$.

Druhý přístup využívá **celou hodnotu** $f(\gamma)$, první jen jeden bit informace („je bod pod grafem?"). Z Rao–Blackwellova principu plyne, že průměr nemůže mít větší rozptyl, protože je to střední hodnota trefa/netrefa přes $\gamma_2$ při pevném $\gamma_1$.

In [ ]:
def hit_or_miss(N):
    return sum(random.random() < f(random.random()) for _ in range(N)) / N


def sample_mean(N):
    return sum(f(random.random()) for _ in range(N)) / N


# přesné rozptyly jednoho příspěvku
var_hm = Z * (1 - Z)
int_f2 = math.sqrt(math.pi / 8) * math.erf(math.sqrt(2))  # integral e^{-2x^2} na [0,1]
var_mean = int_f2 - Z ** 2
print(f"trefa/netrefa: Var = {var_hm:.4f}")
print(f"průměr:        Var = {var_mean:.4f}   (poměr {var_hm / var_mean:.1f}x)")

Průměr má asi $4.7\times$ menší rozptyl, což je totéž jako $4.7\times$ méně vzorků pro stejnou přesnost. A přitom je to **zadarmo**: stejný kód, jen jiný odhadovač.

## 3. Antitetické proměnné

Je-li $f$ monotónní (zde klesající), jsou hodnoty $f(\gamma)$ a $f(1-\gamma)$ **záporně korelované**: velké $\gamma$ dává malé $f$ a naopak. Jejich průměr má pak menší rozptyl než průměr dvou nezávislých:

$$D\frac{f(\gamma)+f(1-\gamma)}{2} = \frac{D f(\gamma)}{2}\,\bigl(1 + \operatorname{Corr}(f(\gamma), f(1-\gamma))\bigr).$$

Při stejném počtu vyhodnocení použijeme $N/2$ dvojic $(\gamma, 1-\gamma)$.

In [ ]:
def antitheticky(N):
    s = 0.0
    for _ in range(N // 2):
        u = random.random()
        s += f(u) + f(1 - u)
    return s / N

## 4. Kontrolní proměnná

Funkci $f(x) = e^{-x^2}$ umíme nahradit blízkou funkcí, jejíž integrál známe přesně: třeba $h(x) = 1 - x^2$ (první dva členy Taylorova rozvoje) s $\int_0^1 h = \tfrac23$. Pak

$$I = \underbrace{\int_0^1 \bigl(f - c\,h\bigr)}_{\text{odhadneme MC}} + c\cdot\tfrac23,$$

tedy odhadovač $f(\gamma) - c\,\bigl(h(\gamma) - \tfrac23\bigr)$. Jeho střední hodnota je pořád $I$ (odečítáme člen s nulovou střední hodnotou), ale pokud je $h$ blízko $f$, je rozdíl skoro konstantní a **rozptyl je malý**.

Optimální volba je $c^* = \dfrac{\operatorname{Cov}(f(\gamma), h(\gamma))}{D\, h(\gamma)}$. Nejjednodušší je $c = 1$ (prostě odečíst $h$), ale optimum bývá jinde: v testu uvidíme obě varianty.

In [ ]:
h = lambda x: 1 - x * x
Eh = 2 / 3


def kontrolni(N, c):
    s = 0.0
    for _ in range(N):
        u = random.random()
        s += f(u) - c * (h(u) - Eh)
    return s / N


# optimální c odhadneme z pilotního vzorku (oddělený od ostré simulace, aby nevnesl vychýlení)
pilot = [random.random() for _ in range(50_000)]
fv = [f(u) for u in pilot]
hv = [h(u) for u in pilot]
mf, mh = sum(fv) / len(fv), sum(hv) / len(hv)
cov = sum((a - mf) * (b - mh) for a, b in zip(fv, hv)) / (len(fv) - 1)
var_h = sum((b - mh) ** 2 for b in hv) / (len(hv) - 1)
c_opt = cov / var_h
print(f"optimální c ≈ {c_opt:.3f}")
corr = cov / math.sqrt(var_h * sum((a - mf) ** 2 for a in fv) / (len(fv) - 1))
print(f"korelace f(γ) a h(γ) ≈ {corr:.4f}")
print(f"očekávané zmenšení rozptylu při c*: 1 / (1 - corr^2) ≈ {1 / (1 - corr ** 2):.0f}x vůči průměru")

Proč kontrolní proměnná funguje, je vidět z obrázku: vlevo mají $f$ a $h$ podobný tvar, takže jejich rozdíl je hladký a kolísá mnohem méně než samotné $f$ (nejde o to, že by byl malý, ale že má malý rozptyl), vpravo dvojice $(h(\gamma), f(\gamma))$ leží téměř na přímce. Sklon této přímky je právě $c^*$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
gr = [i / 200 for i in range(201)]
ax = axes[0]
ax.plot(gr, [f(x) for x in gr], "k", lw=2, label="$f(x) = e^{-x^2}$")
ax.plot(gr, [h(x) for x in gr], "b--", lw=2, label="$h(x) = 1 - x^2$ (známý integrál $2/3$)")
ax.plot(gr, [f(x) - h(x) for x in gr], "g", lw=2, label="$f - h$ (hladký, málo kolísá)")
ax.set_title("Funkce $f$, kontrolní $h$ a jejich rozdíl")
ax.legend(fontsize=8)

ax = axes[1]
ax.scatter(hv[:1500], fv[:1500], s=4, alpha=0.5)
hh = [min(hv), max(hv)]
ax.plot(hh, [mf + c_opt * (t - mh) for t in hh], "r", label=f"sklon $c^* = {c_opt:.2f}$")
ax.set_xlabel("$h(\\gamma)$")
ax.set_ylabel("$f(\\gamma)$")
ax.set_title(f"$f(\\gamma)$ vs $h(\\gamma)$: korelace {corr:.3f}")
ax.legend()
plt.tight_layout()
plt.show()

Korelace je téměř $1$, ale $c^* \approx 0.67$, nikoli $1$: $h$ se mění víc než $f$ (od $1$ do $0$, kdežto $f$ jen od $1$ do $0.37$), takže ji je třeba „zeslabit". Při optimálním $c^*$ se rozptyl zmenší o faktor $1 - \operatorname{Corr}^2$, tedy zhruba šedesátinásobně (u této úlohy). Zvolit $c$ špatně stojí hodně: $c = 1$ dá jen asi čtyřnásobné zlepšení.

## 5. Stratifikace

Čistá náhoda se chová hrbolatě: v jednom úseku $[0,1]$ je bodů víc, v jiném méně (a „mezery" jsou přesně to, co zvyšuje rozptyl). **Stratifikace**: rozdělíme $[0,1]$ na $N$ stejných úseků (strata) a do každého dáme **jeden** náhodný bod:

$$x_i = \frac{i + \gamma_i}{N},\quad i = 0,\dots,N-1.$$

Každý bod je pořád rovnoměrný na svém úseku a odhad je pořád nestranný, ale body jsou rozloženy rovnoměrněji. Rozptyl odhadu pak klesá výrazně rychleji než $1/N$.

In [ ]:
def stratifikace(N):
    return sum(f((i + random.random()) / N) for i in range(N)) / N

Jak se liší **rozložení bodů**: tři způsoby, jak vybrat $N = 20$ bodů z $[0, 1]$. Náhodné body se shlukují a nechávají mezery. Antitetické páry jsou symetrické kolem $1/2$ (spojené čarou). Stratifikované body jsou po jednom v každém z $20$ stejných úseků.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))
Nn = 20
ax.scatter([random.random() for _ in range(Nn)], [2] * Nn, color="tab:blue")
for i in range(Nn // 2):
    u = random.random()
    yy = 0.6 + 0.8 * i / (Nn // 2 - 1)   # páry mírně nad sebou, aby byly vidět
    ax.plot([u, 1 - u], [yy, yy], "o-", color="tab:orange", alpha=0.8, ms=4, lw=1)
ax.scatter([(i + random.random()) / Nn for i in range(Nn)], [0] * Nn, color="tab:green")
for i in range(Nn + 1):
    ax.plot([i / Nn] * 2, [-0.25, 0.25], color="gray", lw=0.6)
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(["stratifikace", "antitetické páry", "čistá náhoda"])
ax.set_ylim(-0.5, 2.5)
ax.set_xlim(0, 1)
ax.set_title(f"{Nn} bodů v [0, 1]: stejný počet, jiné rozložení")
plt.tight_layout()
plt.show()

## 6. Srovnání experimentem

Každou metodu opakujeme $R = 200\times$ pro několik hodnot $N$ a z odhadů spočteme směrodatnou odchylku. Všechny metody dostanou **stejný počet vyhodnocení $f$** ($N$); trefa/netrefa navíc spotřebuje dvě náhodná čísla na bod.

In [ ]:
metody = {
    "trefa/netrefa": hit_or_miss,
    "průměr": sample_mean,
    "antitetické": antitheticky,
    "kontrolní, c = 1": lambda N: kontrolni(N, 1.0),
    "kontrolní, c = c*": lambda N: kontrolni(N, c_opt),
    "stratifikace": stratifikace,
}

R = 200
Ns = [100, 300, 1000, 3000, 10_000]
se_emp = {nazev: [] for nazev in metody}

for N in Ns:
    for nazev, metoda in metody.items():
        odhady = [metoda(N) for _ in range(R)]
        m, se_m = estimate(odhady)
        # nestrannost: průměr R odhadů musí sedět na přesnou hodnotu (se_m je chyba tohoto průměru)
        assert abs(m - Z) < 5 * se_m + 1e-12, (nazev, N, m)
        se_emp[nazev].append(math.sqrt(sum((x - m) ** 2 for x in odhady) / (R - 1)))

print(f"{'metoda':20s}" + "".join(f"{'N=' + str(N):>12s}" for N in Ns))
for nazev, ses in se_emp.items():
    print(f"{nazev:20s}" + "".join(f"{s:12.2e}" for s in ses))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

ax = axes[0]
for nazev, ses in se_emp.items():
    ax.loglog(Ns, ses, "o-", label=nazev)
ref = se_emp["průměr"][0]
ax.loglog(Ns, [ref * (Ns[0] / N) ** 0.5 for N in Ns], "k:", label="sklon $-1/2$ (obyčejné MC)")
ax.loglog(Ns, [se_emp["stratifikace"][0] * (Ns[0] / N) ** 1.5 for N in Ns], "k--", label="sklon $-3/2$")
ax.set_xlabel("počet bodů $N$")
ax.set_ylabel("směrodatná chyba odhadu (z $R = 200$ opakování)")
ax.set_title("Chyba odhadu $I$ podle metody")
ax.legend(fontsize=7, loc="lower left", framealpha=0.9)

# vpravo: kolikrát méně vzorků potřebuje metoda oproti trefa/netrefa při N = 1000
i = Ns.index(1000)
base = se_emp["trefa/netrefa"][i] ** 2
poradi = list(metody)
zisk = [base / se_emp[n][i] ** 2 for n in poradi]
axes[1].barh(poradi, zisk, color="tab:blue")
axes[1].set_xscale("log")
axes[1].set_xlabel("zisk = rozptyl(trefa/netrefa) / rozptyl(metoda)")
axes[1].set_title("Kolikrát méně vzorků potřebujeme (N = 1000)")
axes[1].margins(x=0.35)
for k, v in enumerate(zisk):
    popis = f" {v:,.0f}x".replace(",", " ") if v >= 100 else f" {v:.3g}x"
    axes[1].text(v, k, popis, va="center")
plt.tight_layout()
plt.show()

Co z obrázku plyne:

- **Obyčejné MC** (trefa/netrefa, průměr) klesá se sklonem $-\tfrac12$ a rozdíl mezi nimi je konstantní faktor.
- **Antitetické proměnné a kontrolní proměnná** posunou celou křivku dolů, ale sklon zůstává $-\tfrac12$: změnila se konstanta $\sigma$. Rozdíl mezi $c = 1$ a $c = c^*$ je dobře vidět.
- **Stratifikace** mění i **sklon**: chyba klesá přibližně jako $N^{-3/2}$ (rozptyl jako $N^{-3}$), takže výhoda roste s $N$.

Stratifikace je takhle silná u hladkých jednorozměrných funkcí. Ve více dimenzích potřebujete strata v každé souřadnici (a těch je exponenciálně mnoho), takže výhoda se zmenšuje. Kontrolní proměnné a vážení podle důležitosti se dimenzí netrápí.

## 7. Vážení podle důležitosti (importance sampling)

Dosud jsme generovali z $R(0,1)$ a měnili odhadovač. Teď změníme **rozdělení, ze kterého generujeme**. Chceme

$$I = \int f(x)\,dx = \int \frac{f(x)}{g(x)}\,g(x)\,dx = E_g\!\left[\frac{f(X)}{g(X)}\right],\qquad X \sim g.$$

Odhadovač je $\dfrac{f(X)}{g(X)}$ s $X \sim g$ (**váha** $w = f/g$). Funguje pro **libovolnou** hustotu $g$, která je kladná všude, kde $f \neq 0$. Rozptyl je malý, když je $g$ **podobná $|f|$**: pak je $f/g$ skoro konstantní. Nejlepší (ale nedostupné) $g$ je úměrná $|f|$, kde by $f/g$ byla přesně konstanta.

### 7.1 Vzácný jev: $P(X > 4)$ pro $X \sim N(0,1)$

Přesně $P(X>4) = 1 - \Phi(4) \approx 3.17\cdot 10^{-5}$. Obyčejné MC čeká v miliónu pokusů jen asi $32$ zásahů; ve $100\,000$ pokusech asi $3$, což je přesně situace z cvičení 2, kde intervalový odhad selhával.

Dobrý návrh: generovat **tam, kde jev nastává**. Vezmeme $g(x) = \varphi(x - 4)$ (normální rozdělení posunuté do bodu $4$). Váha je

$$w(x) = \frac{\varphi(x)}{\varphi(x-4)} = e^{-4x + 8}.$$

In [ ]:
prah = 4.0
exact_tail = 0.5 * math.erfc(prah / math.sqrt(2))  # 1 - Phi(4), stabilní i pro malá čísla
print(f"přesně P(X > {prah:.0f}) = {exact_tail:.4e}")

N = 100_000

# obyčejné MC
zasahy = [1.0 if random.gauss(0, 1) > prah else 0.0 for _ in range(N)]
m_plain, se_plain = estimate(zasahy)
print(f"obyčejné MC:   {m_plain:.3e} +- {se_plain:.1e}   (zásahů: {int(sum(zasahy))})")

# vážení podle důležitosti: x ~ N(4, 1), příspěvek = 1[x > 4] * exp(-4 x + 8)
def prispevek_is():
    x = random.gauss(prah, 1)
    return math.exp(-prah * x + prah * prah / 2) if x > prah else 0.0


vz = [prispevek_is() for _ in range(N)]
m_is, se_is = estimate(vz)
check("vážení podle důležitosti", m_is, se_is, exact_tail)
print(f"zlepšení rozptylu: {(se_plain / se_is) ** 2:.0f}x")

Při stejném $N$ je odhad **několik tisíckrát** přesnější. Obyčejné MC vidí jen pár zásahů a odhad je velmi nespolehlivý; u vážení je polovina bodů (ty s $x > 4$) užitečná.

Podívejme se na obrázek: kde leží hustota $\varphi$, kde proposal $g$, a jak se odhady chovají při rostoucím $N$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

xs_ = [i / 50 - 1 for i in range(0, 451)]
phi = lambda x: math.exp(-x * x / 2) / math.sqrt(2 * math.pi)
axes[0].semilogy(xs_, [phi(x) for x in xs_], label=r"cílová $\varphi(x)$ ($N(0,1)$)")
axes[0].semilogy(xs_, [phi(x - prah) for x in xs_], "--", label=r"návrh $g(x) = \varphi(x-4)$")
axes[0].axvline(prah, color="gray", lw=0.8)
axes[0].set_ylim(1e-12, 1)
axes[0].set_title(r"Cílové rozdělení a návrh (svislá čára $x = 4$)")
axes[0].legend()

# běžící odhad
def bezici(hodnoty):
    s, out = 0.0, []
    for i, v in enumerate(hodnoty, 1):
        s += v
        out.append(s / i)
    return out

ax = axes[1]
idx = list(range(1, N + 1))
ax.loglog(idx, [v if v > 0 else None for v in bezici(zasahy)], alpha=0.8, label="obyčejné MC (nula = nezobrazeno)")
ax.loglog(idx, bezici(vz), alpha=0.8, label="vážení podle důležitosti")
ax.axhline(exact_tail, color="k", ls=":", label="přesná hodnota")
ax.set_xlabel("počet bodů")
ax.set_title("Běžící odhad $P(X > 4)$")
ax.legend()
plt.tight_layout()
plt.show()

Vlevo je vidět, že návrh $g$ leží přesně tam, kde je jev. Vpravo obyčejné MC dlouho dává nulu a pak skáče po násobcích $1/N$, zatímco vážený odhad se rychle usadí u přesné hodnoty.

### 7.2 Singularita: $\int_0^1 \dfrac{\cos x}{\sqrt x}\,dx$

Integrand jde pro $x \to 0$ do nekonečna. Obyčejné MC s $R(0,1)$ funguje (integrál konverguje), ale $D\bigl(f(\gamma)\bigr) = \int \dfrac{\cos^2 x}{x}\,dx = \infty$. Pro takový odhad CLV v obvyklé podobě neplatí, chyba je velká a směrodatná chyba spočítaná ze vzorku je nespolehlivá.

Vezmeme $g(x) = \dfrac{1}{2\sqrt x}$ na $(0,1]$. Hustota má **stejnou singularitu** jako integrand, takže ji váha $w = f/g = 2\cos x$ zruší. Generovat z ní umíme inverzní transformací: $G(x) = \sqrt x$, tedy $X = \gamma^2$.

In [ ]:
# přesná hodnota: substituce x = t^2 dává 2 * integral cos(t^2) dt na [0,1], Simpsonovo pravidlo
def simpson(fun, a, b, n=2000):
    hh = (b - a) / n
    s = fun(a) + fun(b)
    for i in range(1, n):
        s += (4 if i % 2 else 2) * fun(a + i * hh)
    return s * hh / 3


I_sing = 2 * simpson(lambda t: math.cos(t * t), 0.0, 1.0)
print(f"přesně I = {I_sing:.8f}")


def plain_sing():
    u = 1.0 - random.random()  # (0, 1], vyhne se dělení nulou
    return math.cos(u) / math.sqrt(u)


def is_sing():
    x = (1.0 - random.random()) ** 2  # X ~ g(x) = 1 / (2 sqrt(x))
    return 2 * math.cos(x)


R = 200
print(f"{'N':>8s} {'obyčejné MC':>16s} {'vážení':>14s}")
for N in (1000, 10_000):
    se_p = estimate([sum(plain_sing() for _ in range(N)) / N for _ in range(R)])[1] * math.sqrt(R)
    se_i = estimate([sum(is_sing() for _ in range(N)) / N for _ in range(R)])[1] * math.sqrt(R)
    print(f"{N:8d} {se_p:16.2e} {se_i:14.2e}")

check("vážení, N = 100000", *estimate([is_sing() for _ in range(100_000)]), I_sing)

(Tabulka ukazuje skutečnou směrodatnou chybu odhadu z $R = 200$ opakování.) Vážení dá výrazně menší chybu a navíc spolehlivě: váha $2\cos x$ je omezená, rozptyl je konečný, a směrodatná chyba spočtená ze vzorku tedy má smysl.

### Pozor na návrh s lehčími chvosty

Vážení má past: $g$ musí mít chvosty **aspoň tak těžké** jako $|f|$ (stejné pravidlo jako u obálky ve cvičení 6). Jinak váha $f/g$ není omezená, rozptyl může být nekonečný a odhad je tiše špatný: většinou vypadá dobře a čas od času přijde obří váha. Dobrá kontrola je podívat se na **největší váhy**: když jedna váha tvoří velkou část součtu, odhadu nevěřte.

## 8. NumPy

Všechny odhadovače jdou psát vektorově:

In [ ]:
try:
    import numpy as np

    rng = np.random.default_rng(1)
    N = 1_000_000

    u = rng.random(N)
    print("průměr:            ", np.mean(np.exp(-u ** 2)))
    print("antitetické:       ", np.mean(np.exp(-u ** 2) + np.exp(-(1 - u) ** 2)) / 2)
    print("kontrolní proměnná:", np.mean(np.exp(-u ** 2) - c_opt * ((1 - u ** 2) - 2 / 3)))
    print("stratifikace:      ", np.mean(np.exp(-((np.arange(N) + rng.random(N)) / N) ** 2)))

    x = rng.normal(4, 1, N)
    w = np.where(x > 4, np.exp(-4 * x + 8), 0.0)
    print(f"P(X > 4): {w.mean():.4e} +- {w.std(ddof=1) / np.sqrt(N):.1e}  (přesně {exact_tail:.4e})")
    print("největší váha tvoří", f"{w.max() / w.sum():.2%} součtu (zdravé: malé číslo)")
    print(f"přesně I = {Z:.6f}")
except ModuleNotFoundError:
    print("NumPy není nainstalované, spusťte:  uv sync")

## Shrnutí

| Metoda | Co dělá | Kdy pomáhá | Pozor |
|--------|---------|------------|-------|
| Průměr místo trefa/netrefa | využije hodnotu $f$, ne bit | vždy, když $f$ umíme vyhodnotit | – |
| Antitetické proměnné | páry $\gamma,\,1-\gamma$ | monotónní $f$ | pro nemonotónní $f$ nemusí pomoci |
| Kontrolní proměnná | odečte blízkou funkci se známým integrálem | existuje $h \approx f$ s $E\,h$ známou | volba $c$; optimum $\operatorname{Cov}/D$ |
| Stratifikace | rovnoměrnější body | hladké $f$, málo dimenzí | ve vysokých dimenzích slábne |
| Vážení podle důležitosti | generuje z $g$, váha $f/g$ | vzácné jevy, singularity | $g$ musí mít těžší chvosty než $f$ |

**Pravidlo:** směrodatná chyba je $\sigma/\sqrt N$. Než přidáte vzorky, zkuste zmenšit $\sigma$, často to jde o řády. Metody lze **kombinovat** (například vážení + stratifikace).

### Zkuste si

1. Odhadněte $\int_0^1 \dfrac{1}{1+x}\,dx = \ln 2$ všemi pěti metodami a porovnejte rozptyly. Je kontrolní proměnná $h(x) = 1 - x/2$ dobrá? Jaké je $c^*$?
2. U vzácného jevu změňte posun návrhu z $4$ na $3$ a na $5$. Který posun dává nejmenší rozptyl a proč?
3. Odhadněte $P(X > 4)$ pomocí návrhu $g(x) = e^{-(x-4)}$ pro $x>4$ (posunuté exponenciální rozdělení), kde je váha $\varphi(x)/g(x)$ a nikdy nepadne mimo oblast jevu.
4. Spočtěte vzorek vah z 7.1 a sledujte největší váhu při posunu $5$. Je odhad důvěryhodný?